<b>FancyImpute -> Smart imputation for missing values by analysing the whole dataset</b>

In [1]:
import pandas as pd

# for overlapping/noisy datasets
# consider the Random Forest version of this
# => MissForest

# pip install fancyimpute
from fancyimpute import KNN

In [2]:
# load the dataset online
df = pd.read_csv("titanic.csv")
df.dropna(subset=["Embarked"], inplace=True)

In [3]:
df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


In [4]:
# check amount of missing values
df.isna().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         0
dtype: int64

In [5]:
# filter out only certain columns
df = df[['Pclass', "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]]

In [6]:
df

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,3,male,22.0,1,0,7.2500,S
1,1,female,38.0,1,0,71.2833,C
2,3,female,26.0,0,0,7.9250,S
3,1,female,35.0,1,0,53.1000,S
4,3,male,35.0,0,0,8.0500,S
...,...,...,...,...,...,...,...
886,2,male,27.0,0,0,13.0000,S
887,1,female,19.0,0,0,30.0000,S
888,3,female,NaN,1,2,23.4500,S
889,1,male,26.0,0,0,30.0000,C


In [7]:
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})

C:\Users\tuomas.valtanen\AppData\Local\Temp\ipykernel_23208\2923248532.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})


In [8]:
# this makes multiple columns with the variable (Separate for yes/no)
from sklearn.preprocessing import OneHotEncoder
variables = ["Embarked"]

# use encoder
encoder = OneHotEncoder(sparse_output=False).set_output(transform="pandas")
one_hot_encoded = encoder.fit_transform(df[variables]).astype(int)
df = pd.concat([df,one_hot_encoded],axis=1).drop(columns=variables)

In [9]:
# small optimization with one hot encoding
# you can always remove EXACTLY ONE of the new columns
df = df.drop("Embarked_S", axis=1)

In [10]:
df.tail(15)

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked_C,Embarked_Q
876,3,0,20.0,0,0,9.8458,0,0
877,3,0,19.0,0,0,7.8958,0,0
878,3,0,NaN,0,0,7.8958,0,0
879,1,1,56.0,0,1,83.1583,1,0
880,2,1,25.0,0,1,26.0000,0,0
881,3,0,33.0,0,0,7.8958,0,0
882,3,1,22.0,0,0,10.5167,0,0
883,2,0,28.0,0,0,10.5000,0,0
884,3,0,25.0,0,0,7.0500,0,0
885,3,1,39.0,0,5,29.1250,0,1


In [11]:
# let's use FancyImpute and KNN to guesstimate the missing age
df_imputed = KNN(k=3).fit_transform(df)

df_imputed = pd.DataFrame(df_imputed, columns=df.columns)

print("\nOriginal DataFrame, amount of missing values:")
print(df.isnull().sum())
print()

print("Imputed DataFrame, amount of missing values:")
print(df_imputed.isnull().sum())
print()

Imputing row 1/889 with 0 missing, elapsed time: 0.050
Imputing row 101/889 with 1 missing, elapsed time: 0.050
Imputing row 201/889 with 1 missing, elapsed time: 0.051
Imputing row 301/889 with 1 missing, elapsed time: 0.051
Imputing row 401/889 with 0 missing, elapsed time: 0.051
Imputing row 501/889 with 0 missing, elapsed time: 0.051
Imputing row 601/889 with 1 missing, elapsed time: 0.052
Imputing row 701/889 with 0 missing, elapsed time: 0.052
Imputing row 801/889 with 0 missing, elapsed time: 0.052

Original DataFrame, amount of missing values:
Pclass          0
Sex             0
Age           177
SibSp           0
Parch           0
Fare            0
Embarked_C      0
Embarked_Q      0
dtype: int64

Imputed DataFrame, amount of missing values:
Pclass        0
Sex           0
Age           0
SibSp         0
Parch         0
Fare          0
Embarked_C    0
Embarked_Q    0
dtype: int64



c:\Users\tuomas.valtanen\advda2025lecturenotes\AdvancedDataAnalytics2025\.venv\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\tuomas.valtanen\advda2025lecturenotes\AdvancedDataAnalytics2025\.venv\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\tuomas.valtanen\advda2025lecturenotes\AdvancedDataAnalytics2025\.venv\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


In [12]:
df_imputed.tail(15)

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked_C,Embarked_Q
874,3.0,0.0,20.000000,0.0,0.0,9.8458,0.0,0.0
875,3.0,0.0,19.000000,0.0,0.0,7.8958,0.0,0.0
876,3.0,0.0,23.333333,0.0,0.0,7.8958,0.0,0.0
877,1.0,1.0,56.000000,0.0,1.0,83.1583,1.0,0.0
878,2.0,1.0,25.000000,0.0,1.0,26.0000,0.0,0.0
879,3.0,0.0,33.000000,0.0,0.0,7.8958,0.0,0.0
880,3.0,1.0,22.000000,0.0,0.0,10.5167,0.0,0.0
881,2.0,0.0,28.000000,0.0,0.0,10.5000,0.0,0.0
882,3.0,0.0,25.000000,0.0,0.0,7.0500,0.0,0.0
883,3.0,1.0,39.000000,0.0,5.0,29.1250,0.0,1.0
